# 真实数据案例：网络分析流程与 NetworkX 实践

来自《生物医学网络科学》（Biomedical Network Science）。数据文件放在与本笔记本同级的 `data/` 文件夹中。

In [ ]:
# 数据文件（JupyterLite 专用；文件已在时什么也不做）
import os, js
from pyodide.http import pyfetch
_base = str(js.location.href).split("/extensions/")[0].split("/build/")[0] + "/files/practice/data/"
os.makedirs("data", exist_ok=True)
for _f in ['yeast_string_physical_v12_hc.csv', 'yeast_string_physical_v12_nodes.csv']:
    if not os.path.exists("data/" + _f):
        _r = await pyfetch(_base + _f)
        open("data/" + _f, "wb").write(await _r.bytes())


# 酵母蛋白质物理相互作用网络：一次完整的 NetworkX 分析流程

本案例按[网络分析的一般流程](analysis-workflow.workflow.html)走一遍：提出问题 → 获取并清洗数据 → 构网 → 网络概况 → 中心性与社团 → 与 ER、保度零模型比较 → 敏感性分析 → 报告。每一步都把专题页“应记录或检查的内容”打印出来，最后汇总成一张可以直接写进方法和结果部分的表。

**问题**：酵母（*Saccharomyces cerevisiae*）高置信度的蛋白质物理相互作用网络有哪些结构特征——是否局部成团、是否有小世界性、度相关是正还是负、是否由模块组成？哪些蛋白处在中心位置？这些结论在换一个置信度阈值后是否仍然成立？

**1. 数据来源**

- 数据库：STRING v12.0，物种 4932（酿酒酵母），**物理相互作用子网络**文件 `4932.protein.physical.links.v12.0.txt.gz`，以及蛋白信息文件 `4932.protein.info.v12.0.txt.gz`（`https://stringdb-downloads.org/download/`，2026-11-20 下载）。
- 许可证：STRING 的全部数据与下载文件按 **CC BY 4.0** 提供（`https://string-db.org/cgi/access`）。引用：Szklarczyk D 等, The STRING database in 2023: protein–protein association networks and functional enrichment analyses for any sequenced genome of interest, *Nucleic Acids Research* 2023;51:D638–D646, DOI [10.1093/nar/gkac1000](https://doi.org/10.1093/nar/gkac1000)。
- 我们做的修改：只保留综合得分（combined score）≥ 700（STRING 的“高置信度”）的物理相互作用，每对蛋白只保留一行；去掉 STRING 编号前缀 `4932.`；蛋白注释只保留第一句并截断到 80 个字符。文件为 `data/yeast_string_physical_v12_hc.csv`（边）和 `data/yeast_string_physical_v12_nodes.csv`（节点）。
- 注意：STRING 的“物理子网络”表示两个蛋白属于同一物理复合物的证据，**不一定是直接结合**；得分整合了实验、数据库和文本挖掘等多个证据通道。

## 2. 读入与清洗：先看数据，再算指标

在计算任何指标之前，先检查：行数、得分范围、自环、重复的蛋白对、标识符能否映射到基因名、有没有映射失败。

In [ ]:
import time
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

t0 = time.time()
edges = pd.read_csv("data/yeast_string_physical_v12_hc.csv")
info = pd.read_csv("data/yeast_string_physical_v12_nodes.csv").set_index("id")
print("边表行数 %d；得分范围 %d–%d" % (len(edges), edges.combined_score.min(), edges.combined_score.max()))
print("自环 %d 条" % (edges.protein1 == edges.protein2).sum())
pair = edges[["protein1", "protein2"]].apply(lambda s: tuple(sorted(s)), axis=1)
print("重复的无序蛋白对 %d 个" % pair.duplicated().sum())
ids = set(edges.protein1) | set(edges.protein2)
missing = ids - set(info.index)
print("出现的蛋白 %d 个，在信息表中找不到的 %d 个" % (len(ids), len(missing)))
name = info.preferred_name.to_dict()
dup_names = info.preferred_name[info.preferred_name.duplicated(keep=False)]
print("基因名重复（一名对多个编号）的条目 %d 个" % len(dup_names))
print(edges.head(3).assign(name1=lambda d: d.protein1.map(name), name2=lambda d: d.protein2.map(name)).to_string(index=False))

## 3. 构网

节点为蛋白（STRING 编号，打印时换成基因名），边为无向、加权。权重的含义要写清楚：`weight` = 得分/1000，是**相似度**（越大越可信）；路径类算法需要**距离**，另存一个属性 `distance` $=-\log w$（专题页“权重是相似度还是距离”）。本案例的主体分析按无权图进行，加权版本只在中心性一步作对照。

连通性检查：报告连通分支数、最大连通分支（巨分支）占比，并说明后续路径类指标只在巨分支上计算、丢掉了多少节点。

In [ ]:
G = nx.Graph()
for a, b, s in edges.itertuples(index=False):
    w = s / 1000
    G.add_edge(a, b, weight=w, distance=-np.log(w))
comps = sorted(nx.connected_components(G), key=len, reverse=True)
gc = G.subgraph(comps[0]).copy()
N, M = G.number_of_nodes(), G.number_of_edges()
print("全网: N = %d, M = %d, 密度 = %.5f, 平均度 = %.2f" % (N, M, nx.density(G), 2 * M / N))
print("连通分支 %d 个；巨分支 %d 个节点（%.1f%%）、%d 条边；第二大分支 %d 个节点；孤立点 %d"
      % (len(comps), len(comps[0]), 100 * len(comps[0]) / N, gc.number_of_edges(), len(comps[1]), nx.number_of_isolates(G)))
size_counts = pd.Series([len(c) for c in comps[1:]]).value_counts().sort_index()
print("非巨分支的大小分布（大小: 个数）:", dict(size_counts.head(8)))

## 4. 网络概况与核心指标

在巨分支上计算：度分布、平均聚集系数 $C$、平均最短路径 $L$、直径的下界、度同配系数 $r$。平均最短路径的精确计算需要从每个节点做一次广度优先搜索，这里从固定种子下随机抽取的 300 个源节点估计（专题页“大规模网络的计算策略”中的近似做法），并用两组不同的源节点检查估计是否稳定。

In [ ]:
rng = np.random.default_rng(2026)
deg = np.array([d for _, d in gc.degree()])

def mean_path(H, k=300, seed=0):
    r_ = np.random.default_rng(seed)
    nodes = list(H)
    tot = cnt = ecc = 0
    for i in r_.choice(len(nodes), min(k, len(nodes)), replace=False):
        d = nx.single_source_shortest_path_length(H, nodes[i])
        tot += sum(d.values()); cnt += len(d) - 1; ecc = max(ecc, max(d.values()))
    return tot / cnt, ecc

C_real = nx.average_clustering(gc)
T_real = nx.transitivity(gc)
L_real, ecc_real = mean_path(gc, seed=0)
L_check, _ = mean_path(gc, seed=1)
r_real = nx.degree_assortativity_coefficient(gc)
print("度: 最小 %d, 中位数 %d, 平均 %.1f, 最大 %d；<k^2>/<k> = %.1f" % (deg.min(), np.median(deg), deg.mean(), deg.max(), (deg**2).mean() / deg.mean()))
print("平均聚集系数 C = %.3f；传递性（全局聚集系数）= %.3f" % (C_real, T_real))
print("平均最短路径 L ≈ %.2f（另一组源节点 %.2f）；直径 ≥ %d" % (L_real, L_check, ecc_real))
print("度同配系数 r = %.3f" % r_real)

## 5. 中心性与社团

**中心性**：度中心性、介数中心性（Brandes 算法；为节省时间用 250 个抽样源节点近似，`seed` 固定）和加权介数（用 `distance` 作为路径长度）。比较几种中心性给出的前 10 名，并用 Spearman 秩相关看它们整体是否一致。

**社团**：Louvain 算法有随机性，专题页把“只跑一次”列为常见遗漏。这里用 5 个不同种子各跑一次，报告模块度 $Q$ 的范围，并用种子 0 的结果展示最大的几个社团（列出度最高的成员和它们的注释）。

In [ ]:
bc = nx.betweenness_centrality(gc, k=250, seed=1)
bcw = nx.betweenness_centrality(gc, k=250, seed=1, weight="distance")
cent = pd.DataFrame({"degree": dict(gc.degree()), "betweenness": bc, "betweenness_w": bcw})
cent["name"] = cent.index.map(name)
cent["annotation"] = cent.index.map(info.annotation.to_dict())
for col in ["degree", "betweenness"]:
    print("== %s 前 10 名" % col)
    print(cent.sort_values(col, ascending=False).head(10)[["name", "degree", "betweenness", "annotation"]].to_string(index=False, float_format="%.4f"))
rho = cent[["degree", "betweenness", "betweenness_w"]].rank().corr()
print("中心性之间的 Spearman 秩相关:\n", rho.round(2).to_string())
overlap = len(set(cent.nlargest(50, "degree").index) & set(cent.nlargest(50, "betweenness").index))
print("度前 50 与介数前 50 的重合: %d 个" % overlap)

Qs, ncs = [], []
for s in range(5):
    cs = nx.community.louvain_communities(gc, seed=s)
    Qs.append(nx.community.modularity(gc, cs)); ncs.append(len(cs))
    if s == 0:
        comms = sorted(cs, key=len, reverse=True)
print("Louvain 5 个种子: Q = %.3f–%.3f，社团数 %d–%d" % (min(Qs), max(Qs), min(ncs), max(ncs)))
for i, c in enumerate(comms[:8]):
    sub = cent.loc[list(c)].sort_values("degree", ascending=False)
    inside = gc.subgraph(c).number_of_edges()
    print("社团 %d: %d 个蛋白，内部边 %d；度最高的成员: %s" % (i, len(c), inside, ", ".join(sub.name.head(6))))
    print("    其注释: %s | %s" % (sub.annotation.iloc[0], sub.annotation.iloc[1]))

## 6. 与零模型比较

按专题页“用模型网络与零模型解读真实网络”的表格，比较真实网络与两种零模型（为了在浏览器里也能较快运行，各只生成 5 个，固定种子）：

- **ER 随机图** $G(N,M)$：与巨分支相同的节点数和边数（指标在其巨分支上计算）；
- **保度随机化**：双边交换 $3M$ 次，每个节点的度不变（`nx.double_edge_swap`；ER 与保度网络都保持简单图）。

指标：$C$、$L$、$r$、Louvain 模块度 $Q$，以及以 ER 为参照的小世界系数 $\sigma=\dfrac{C/C_{\rm rand}}{L/L_{\rm rand}}$。零模型次数少，所以只报告均值、标准差和 $z$ 分数；聚集系数用稀疏矩阵公式 $C_i=(A^3)_{ii}/[k_i(k_i-1)]$ 计算，先检查它与 NetworkX 的结果一致。

In [ ]:
n_gc, m_gc = gc.number_of_nodes(), gc.number_of_edges()

from scipy.sparse import csr_matrix

def fast_clustering(H):
    """平均局部聚集系数：C_i = (A^3)_ii / (k_i (k_i - 1))，用稀疏矩阵计算（度 < 2 的节点记为 0，与 NetworkX 相同）。"""
    A = nx.to_scipy_sparse_array(H, format="csr", weight=None)
    k = np.asarray(A.sum(1)).ravel()
    tri2 = np.asarray((A @ A).multiply(A).sum(1)).ravel()      # = 2 × 经过 i 的三角形数
    c = np.where(k > 1, tri2 / np.maximum(k * (k - 1), 1), 0.0)
    return c.mean()

assert abs(fast_clustering(gc) - C_real) < 1e-9, "稀疏实现应与 nx.average_clustering 一致"

def summary(H):
    H = H.subgraph(max(nx.connected_components(H), key=len)).copy()
    return dict(C=fast_clustering(H), L=mean_path(H, k=100, seed=0)[0],
                r=nx.degree_assortativity_coefficient(H),
                Q=nx.community.modularity(H, nx.community.louvain_communities(H, seed=0)))

real = dict(C=C_real, L=L_real, r=r_real, Q=Qs[0])
null = {"ER": [], "degree-preserving": []}
R_null = 5
for b in range(R_null):
    null["ER"].append(summary(nx.gnm_random_graph(n_gc, m_gc, seed=int(rng.integers(1 << 31)))))
    H = gc.copy()
    nx.double_edge_swap(H, nswap=3 * m_gc, max_tries=50 * m_gc, seed=int(rng.integers(1 << 31)))
    null["degree-preserving"].append(summary(H))
rows = []
for k in ["C", "L", "r", "Q"]:
    row = {"metric": k, "real": real[k]}
    for nm, lst in null.items():
        x = np.array([d[k] for d in lst])
        row[nm + " mean"] = x.mean(); row[nm + " sd"] = x.std(ddof=1)
        row[nm + " z"] = (real[k] - x.mean()) / x.std(ddof=1)
    rows.append(row)
comp = pd.DataFrame(rows).set_index("metric")
print(comp.round(3).to_string())
C_er = comp.loc["C", "ER mean"]; L_er = comp.loc["L", "ER mean"]
sigma = (C_real / C_er) / (L_real / L_er)
print("小世界系数 sigma = (C/C_rand)/(L/L_rand) = (%.3f/%.4f)/(%.2f/%.2f) = %.1f" % (C_real, C_er, L_real, L_er, sigma))

**敏感性分析：换一个置信度阈值。** 专题页提醒，“STRING 置信度 0.4 还是 0.7”这类选择要做敏感性分析。本案例的数据文件只包含 ≥ 700 的边，因此只能向更严格的方向检查：把阈值提高到 900（“最高置信度”），重复概况指标，看主要结论（高聚集、正同配、高模块度、路径长于随机网络）是否仍然成立。

In [ ]:
G9 = nx.Graph([(a, b) for a, b, d in G.edges(data=True) if d["weight"] >= 0.9])
gc9 = G9.subgraph(max(nx.connected_components(G9), key=len)).copy()
s9 = summary(gc9)
H9 = gc9.copy()
nx.double_edge_swap(H9, nswap=5 * H9.number_of_edges(), max_tries=50 * H9.number_of_edges(), seed=7)
s9n = summary(H9)
print("阈值 900: 全网 N = %d, M = %d；巨分支 %d 个节点（%.1f%%）"
      % (G9.number_of_nodes(), G9.number_of_edges(), gc9.number_of_nodes(), 100 * gc9.number_of_nodes() / G9.number_of_nodes()))
for k in ["C", "L", "r", "Q"]:
    print("  %s: 阈值 700 为 %.3f，阈值 900 为 %.3f（一个保度随机网络: %.3f）" % (k, real[k], s9[k], s9n[k]))
top9 = sorted(gc9.degree, key=lambda x: -x[1])[:10]
top7 = set(cent.nlargest(10, "degree").index)
print("度前 10 名在两个阈值下的重合: %d 个" % len(top7 & {a for a, _ in top9}))

## 7. 作图

左上：度的互补累积分布（双对数坐标，只作描述，不据此断言“无标度”）。右上：局部聚集系数随度的变化 $C(k)$。左下：真实网络与两种零模型的指标比较。右下：最大的 8 个社团中内部边最少的一个的子图（便于看清结构；节点大小按度，标注度最高的几个蛋白）。

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
ax = axes[0, 0]
ks = np.sort(deg)
ax.loglog(ks, 1 - np.arange(len(ks)) / len(ks), ".", ms=3)
ax.set_xlabel("degree k"); ax.set_ylabel("P(K >= k)"); ax.set_title("Degree CCDF (giant component, N = %d)" % n_gc)

ax = axes[0, 1]
cl = nx.clustering(gc)
dd = pd.DataFrame({"k": dict(gc.degree()), "c": cl})
bins = np.unique(np.logspace(0, np.log10(dd.k.max()), 15).astype(int))
dd["bin"] = pd.cut(dd.k, np.r_[bins, dd.k.max() + 1], right=False)
mb = dd.groupby("bin", observed=True).agg(k=("k", "mean"), c=("c", "mean"))
ax.semilogx(dd.k, dd.c, ".", ms=2, alpha=0.3, color="0.5")
ax.semilogx(mb.k, mb.c, "o-", color="tab:red", label="binned mean")
ax.set_xlabel("degree k"); ax.set_ylabel("local clustering C(k)"); ax.legend(); ax.set_title("Clustering vs degree")

ax = axes[1, 0]
metrics = ["C", "L", "r", "Q"]
x = np.arange(len(metrics))
ax.bar(x - 0.25, [real[k] for k in metrics], 0.25, label="real")
for j, nm in enumerate(["ER", "degree-preserving"]):
    ax.bar(x + 0.25 * j, comp[nm + " mean"], 0.25, yerr=comp[nm + " sd"], label=nm)
ax.set_xticks(x); ax.set_xticklabels(["clustering C", "mean path L", "assortativity r", "modularity Q"])
ax.axhline(0, color="k", lw=0.6); ax.legend(); ax.set_title("Real network vs null models (%d each)" % R_null)

ax = axes[1, 1]
c = min(comms[:8], key=lambda c: gc.subgraph(c).number_of_edges())   # 前 8 个社团中内部边最少的一个，便于看清
S = gc.subgraph(c)
pos = nx.spring_layout(S, seed=2)
dS = dict(S.degree())
nx.draw_networkx_edges(S, pos, ax=ax, alpha=0.15)
nx.draw_networkx_nodes(S, pos, ax=ax, node_size=[5 + 2 * dS[v] for v in S], node_color="tab:blue", alpha=0.8)
topS = sorted(dS, key=dS.get, reverse=True)[:8]
nx.draw_networkx_labels(S, pos, ax=ax, labels={v: name[v] for v in topS}, font_size=7)
ax.set_title("One Louvain community: %d proteins, %d edges" % (S.number_of_nodes(), S.number_of_edges())); ax.axis("off")
plt.tight_layout()

## 报告：可以直接写进论文的记录

下面把专题页表格要求记录的内容汇总打印出来。

In [ ]:
report = {
    "数据": "STRING v12.0 physical links, S. cerevisiae (4932), CC BY 4.0, 下载于 2026-11-20",
    "构网": "无向；combined score >= 700；无自环、无重边；权重 w = score/1000，距离 -log w",
    "全网规模": "N = %d, M = %d, 连通分支 %d 个" % (N, M, len(comps)),
    "巨分支": "%d 个节点（%.1f%%），后续指标均在巨分支上计算" % (n_gc, 100 * n_gc / N),
    "聚集与路径": "C = %.3f, L ≈ %.2f（300 个抽样源节点）" % (C_real, L_real),
    "度相关": "r = %.3f" % r_real,
    "社团": "Louvain，5 个种子，Q = %.3f–%.3f" % (min(Qs), max(Qs)),
    "零模型": "ER 与保度随机化各 %d 个；C 的 z 分数 %.0f（ER）、%.0f（保度）"
              % (R_null, comp.loc["C", "ER z"], comp.loc["C", "degree-preserving z"]),
    "敏感性": "阈值 900：C = %.3f, r = %.3f, Q = %.3f" % (s9["C"], s9["r"], s9["Q"]),
    "软件": "networkx %s, numpy %s, pandas %s" % (nx.__version__, np.__version__, pd.__version__),
}
for k, v in report.items():
    print("%-6s %s" % (k, v))
print("总运行时间 %.1f 秒" % (time.time() - t0))

## 8. 解读与局限

**结果怎么读**（具体数字以上面的运行输出为准）：

- **高度局部成团**：平均聚集系数约 0.6，远高于 ER 和保度随机网络（后者只有约 0.1）。保度零模型并没有把 $C$ 拉近真实值，说明这种成团不是由几个枢纽造成的，而是来自网络本身的组织方式。
- **路径并不短**：平均最短路径比两种随机网络都长得多，所以尽管以 ER 为参照算出的 $\sigma$ 远大于 1，按“$L\approx L_{\rm rand}$ 且 $C\gg C_{\rm rand}$”的标准，这个网络并不是典型的小世界网络。$\sigma$ 只是一个比值，大的 $C/C_{\rm rand}$ 会掩盖 $L/L_{\rm rand}$ 大于 1 的事实，报告时应把两项分开写出。
- **正的度同配**：专题页的表格提示生物网络的度同配“常为负”，这里却明显为正，而且保度随机化后变为接近 0 或略负。原因在于 STRING 的物理子网络记录的是“同属一个复合物”的证据：大型复合物（如核糖体、蛋白酶体）的成员两两相连，高度节点彼此相连。这说明度相关的符号取决于**边的定义**，不能脱离数据来源泛泛而谈。
- **模块化**：$Q$ 在不同种子之间很稳定，并显著高于两种零模型；最大的几个社团中度最高的成员往往属于同一个复合物或同一个细胞过程（输出中可以看到以 40S 核糖体蛋白、Sm 小核核糖核蛋白、SNARE 蛋白、核仁 rRNA 加工蛋白为首的社团），这是“模块 ↔ 功能”的直观证据，但要正式下结论需要做功能富集检验。
- **关于 $z$ 分数**：表中的 $z$ 值动辄上百，只说明零分布非常窄，不宜当作效应大小来比较；报告时应同时给出真实值和零模型均值。
- **中心性**：度最高的 10 个蛋白全是 40S 核糖体亚基的成员——它们的高度来自所在复合物很大，而不是“单个蛋白特别重要”。介数最高的是泛素–核糖体蛋白融合蛋白、肌动蛋白和组蛋白等连接多个模块的蛋白。度与介数的秩相关只是中等（约 0.5），前 50 名只有少数重合，两者回答的是不同问题；加权与无权介数的排序也不完全相同，需要写明用的是哪一种。
- **敏感性**：阈值提高到 900 后网络变小，但高聚集、正同配和高模块度这几个定性结论保持不变。

**局限**：

- STRING 的“物理”边包含复合物共成员关系，不等于二元直接结合；用酵母双杂交数据或 BioGRID 的二元相互作用构网，度相关和聚集系数都会不同。
- 研究偏倚：研究得多的蛋白有更多记录，度和中心性部分反映的是“被研究得多”（见[测量误差、缺失与偏倚](network-construction.bias-and-missingness.html)）。
- 为了运行时间，平均路径和介数用抽样近似，零模型只生成 5 个；正式研究应增加源节点和零模型数目，并报告经验 $p$ 值。
- 敏感性分析只检查了更严格的阈值；放宽到 400 需要重新从 STRING 下载完整文件。